# Credit Card Fraud Detection - Model Training
Training models on highly imbalanced data with proper evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc, roc_curve

import warnings
warnings.filterwarnings('ignore')

## 1. Load & Preprocess

In [ ]:
df = pd.read_csv('../data/creditcard_sampled.csv')

X = df.drop(['TransactionID', 'Class'], axis=1)
y = df['Class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Fraud in Test: {y_test.sum()} ({y_test.mean():.2%})')

## 2. Logistic Regression (Balanced)

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print('Logistic Regression:')
print(classification_report(y_test, lr_pred))

## 3. Random Forest (Balanced)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print('Random Forest:')
print(classification_report(y_test, rf_pred))

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (name, pred) in enumerate([('Logistic Regression', lr_pred), ('Random Forest', rf_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 5. Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in [('Logistic Regression', lr), ('Random Forest', rf)]:
    y_scores = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_scores)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, label=f'{name} (AUC={pr_auc:.3f})', linewidth=2)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in [('Logistic Regression', lr), ('Random Forest', rf)]:
    y_scores = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=['Amount', 'V1', 'V2', 'V3', 'V4', 'V5']).sort_values()

plt.figure(figsize=(8, 5))
plt.barh(importances.index, importances.values, color='#9b59b6')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Save Best Model

In [ ]:
with open('../models/fraud_model.pkl', 'wb') as f:
    pickle.dump(rf, f)
with open('../models/fraud_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model saved!')